# Concept Direction Experiment Harness

Unified notebook harness for concept-direction experimentation across Gemma 2 and Gemma 3 model variants.

This notebook is intended to be executed manually or via papermill. It keeps the original nine experimental phases, but it now manages resources more aggressively by opening fresh model sessions at phase boundaries and immediately offloading retained artifacts to CPU.

## Notebook Parameters

The next code cell is tagged for papermill and should remain the single source of flat experiment parameters.

In [ ]:
# Parameters - These will be injected by papermill during parameterized runs

EXPERIMENT_CONFIG_PATH = "configs/gemma3_4b_it_local_capitals_states.yaml"
EXPERIMENT_CONFIG_NAME = "manual"
EXPERIMENT_NAME = "gemma3_4b_it_local_capitals_states"

VERBOSE = True

In [ ]:
from __future__ import annotations

import gc
import json
import os
import shutil
import sys
from pathlib import Path
from typing import Any

# Reduce CUDA memory fragmentation before importing torch.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import matplotlib.pyplot as plt
import torch


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(f"Could not locate repo root from {start}")


CWD = Path.cwd().resolve()
REPO_ROOT = _find_repo_root(CWD)
TESTS_DIR = REPO_ROOT / "tests"
HARNESS_DIR = TESTS_DIR / "concept_direction_approach_parity"
SHARED_HARNESS_DIR = TESTS_DIR / "nb_experiment_harness"

for path in (REPO_ROOT, TESTS_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from tests.nb_experiment_harness.notebook_bootstrap import bootstrap_notebook_imports

bootstrap_notebook_imports(cwd=CWD, extra_paths=[HARNESS_DIR])

from interpretune.utils import ensure_local_feature_explanations
from it_examples.utils.nb_ui_utils import (
    display_ablation_chart,
    display_key_token_logits,
    display_top_features_comparison,
    display_topk_token_predictions,
)
from tests.concept_direction_approach_parity.concept_direction import (
    build_notebook_harness_config,
    collect_feature_pool as _collect_feature_pool,
    collect_summary,
    compute_embed_direction as _compute_embed_direction,
    compute_store_direction as _compute_store_direction,
    display_direction_probe_results,
    display_tokenizer_target_summary,
    prepare_local_explanation_backfill,
    run_direct_projection_pipeline as _run_direct_projection_pipeline,
    run_direct_projection_scale_sweep as _run_direct_projection_scale_sweep,
    run_pipeline as _run_pipeline,
    run_scale_sweep as _run_scale_sweep,
)
from tests.nb_experiment_harness.nb_harness_utils import (
    display_baseline_path_debug_report,
    display_debug_intervention_validation_report,
    display_initial_sanity_check_report,
    display_local_explanation_report,
    display_tokenizer_verification_report,
    feature_ids_to_tuples,
    log_notebook_config,
    phase_run_name as _phase_run_name,
    tensor_to_cpu,
)
from tests.nb_experiment_harness.pipeline_patterns import (
    collect_baseline_path_debug as _collect_baseline_path_debug,
    run_ablations as _run_ablations,
    run_debug_intervention_validation as _run_debug_intervention_validation,
    run_direction_probes as _run_direction_probes,
    run_initial_sanity_check as _run_initial_sanity_check,
    run_sign_aware as _run_sign_aware,
    run_tokenizer_verification as _run_tokenizer_verification,
)
from tests.nb_experiment_harness.session import experiment_session

print("Imports complete.")

In [ ]:
CONFIG_PATH = Path(EXPERIMENT_CONFIG_PATH).expanduser()
if not CONFIG_PATH.is_absolute():
    CONFIG_PATH = (Path.cwd() / CONFIG_PATH).resolve()

NOTEBOOK_CFG, SHOULD_CLEANUP_WORK_ROOT, RESOLVED_EXPERIMENT_CONFIG = build_notebook_harness_config(CONFIG_PATH)
EXPERIMENT_CONFIG_PATH = str(CONFIG_PATH)
USES_STORE_DIRECTION = NOTEBOOK_CFG.supports_store_direction
IS_DEBUG_INTERVENTION_MODE = NOTEBOOK_CFG.is_debug_intervention_mode
CONFIG_SUMMARY = log_notebook_config(
    NOTEBOOK_CFG,
    config_path=CONFIG_PATH,
    work_root_is_temporary=SHOULD_CLEANUP_WORK_ROOT,
    verbose=VERBOSE,
)

In [ ]:
def print_phase_skip(reason: str) -> None:
    print(f"Skipped for analysis_mode={NOTEBOOK_CFG.analysis_mode}: {reason}")


RESULTS: dict[str, Any] = {}

## Phase 1: Session Initialization and Tokenizer Verification

In [ ]:
tokenizer_report = _run_tokenizer_verification(NOTEBOOK_CFG)
RESULTS["tokenizer_report"] = tokenizer_report

display_tokenizer_verification_report(tokenizer_report)
display_tokenizer_target_summary(tokenizer_report)

In [ ]:
reasoning_check = _run_initial_sanity_check(NOTEBOOK_CFG)
RESULTS["initial_sanity_check"] = reasoning_check

display_initial_sanity_check_report(reasoning_check)

In [ ]:
if NOTEBOOK_CFG.enable_baseline_path_debug:
    baseline_path_debug = _collect_baseline_path_debug(NOTEBOOK_CFG)
    RESULTS["baseline_path_debug"] = baseline_path_debug
    display_baseline_path_debug_report(baseline_path_debug)
else:
    print("Baseline path debug: DISABLED (set ENABLE_BASELINE_PATH_DEBUG=true to enable)")

## Phase 2 and 3: Embed-Based Direction and Full Pipeline

In [ ]:
if IS_DEBUG_INTERVENTION_MODE:
    print_phase_skip("embed-direction pipeline is replaced by debug intervention validation.")
else:
    embed_direction_result = _compute_embed_direction(NOTEBOOK_CFG)
    embed_pipeline = _run_pipeline(
        NOTEBOOK_CFG,
        embed_direction_result["direction"],
        "Embed",
        scale_factor=NOTEBOOK_CFG.default_scale_factor,
        top_n=NOTEBOOK_CFG.top_n,
        group_a_ids=embed_direction_result["group_a_ids"],
        group_b_ids=embed_direction_result["group_b_ids"],
    )
    RESULTS["embed_direction_result"] = embed_direction_result
    RESULTS["embed_pipeline"] = embed_pipeline
    print(f"Embed direction norm: {torch.linalg.vector_norm(embed_direction_result['direction']):.6f}")
    print(
        f"Pre-gap ({embed_pipeline['target_a_tok']}−{embed_pipeline['target_b_tok']}): "
        f"{embed_pipeline['pre_gap']:.4f}"
    )
    print(f"Post-gap: {embed_pipeline['post_gap']:.4f}")
    print(f"Gap Δ: {embed_pipeline['gap_delta']:+.4f}")
    display_top_features_comparison(
        {"Embed — Top Features": embed_pipeline["feature_ids"]},
        {"Embed — Top Features": embed_pipeline["feature_scores"]},
        neuronpedia_model=NOTEBOOK_CFG.neuronpedia_model,
        neuronpedia_set=NOTEBOOK_CFG.neuronpedia_set,
        neuronpedia_base_url=NOTEBOOK_CFG.neuronpedia_base_url,
    )

In [ ]:
if IS_DEBUG_INTERVENTION_MODE:
    print_phase_skip("embed-direction display is replaced by debug intervention validation.")
else:
    with experiment_session(
        NOTEBOOK_CFG.work_root,
        _phase_run_name(NOTEBOOK_CFG.experiment_name, "embed_display"),
        **NOTEBOOK_CFG.session_kwargs,
    ) as (_, _, tokenizer):
        display_topk_token_predictions(
            embed_pipeline["rendered_prompt"],
            embed_pipeline["pre_logits"],
            embed_pipeline["post_logits"],
            tokenizer,
            k=5,
            key_tokens=[
                (label, token_id) for label, token_id in zip(embed_pipeline["key_labels"], embed_pipeline["key_ids"])
            ],
        )
        display_key_token_logits(
            embed_pipeline["pre_logits"],
            embed_pipeline["post_logits"],
            embed_pipeline["key_ids"],
            embed_pipeline["key_labels"],
            title=f"Embed {NOTEBOOK_CFG.default_scale_factor}× — Key-Token Logits",
        )

## Phase 4 and 5: Store-Based Direction and Summary

In [ ]:
if not USES_STORE_DIRECTION:
    print_phase_skip("store-direction pipeline only applies to concept_pair mode.")
else:
    store_direction_result = _compute_store_direction(NOTEBOOK_CFG)
    store_pipeline = _run_pipeline(
        NOTEBOOK_CFG,
        store_direction_result["direction"],
        "Store",
        scale_factor=NOTEBOOK_CFG.default_scale_factor,
        top_n=NOTEBOOK_CFG.top_n,
        group_a_ids=store_direction_result["group_a_ids"],
        group_b_ids=store_direction_result["group_b_ids"],
    )
    RESULTS["store_direction_result"] = store_direction_result
    RESULTS["store_pipeline"] = store_pipeline
    cosine_similarity = float(
        torch.nn.functional.cosine_similarity(
            embed_direction_result["direction"].unsqueeze(0), store_direction_result["direction"].unsqueeze(0)
        ).item()
    )
    embed_set = set(embed_pipeline["feature_ids"])
    store_set = set(store_pipeline["feature_ids"])
    shared = embed_set & store_set
    total = embed_set | store_set
    feature_jaccard = len(shared) / len(total) if total else 0.0
    RESULTS["comparison"] = {
        "cosine_similarity": cosine_similarity,
        "feature_jaccard": feature_jaccard,
        "shared_feature_count": len(shared),
        "total_feature_count": len(total),
    }
    print(f"Store direction norm: {torch.linalg.vector_norm(store_direction_result['direction']):.6f}")
    print(f"Predictions: {store_direction_result['prediction_info']['n_correct']}/{store_direction_result['n_total']}")
    print(f"Cosine similarity: {cosine_similarity:.6f}")
    print(f"Feature Jaccard: {feature_jaccard:.4f} ({len(shared)}/{len(total)})")
    print(f"Embed Δ: {embed_pipeline['gap_delta']:+.4f} | Store Δ: {store_pipeline['gap_delta']:+.4f}")

In [ ]:
if not USES_STORE_DIRECTION:
    print_phase_skip("store-direction display only applies to concept_pair mode.")
else:
    with experiment_session(
        NOTEBOOK_CFG.work_root,
        _phase_run_name(NOTEBOOK_CFG.experiment_name, "store_display"),
        **NOTEBOOK_CFG.session_kwargs,
    ) as (_, _, tokenizer):
        display_top_features_comparison(
            {
                "Embed — Top Features": embed_pipeline["feature_ids"],
                "Store — Top Features": store_pipeline["feature_ids"],
            },
            {
                "Embed — Top Features": embed_pipeline["feature_scores"],
                "Store — Top Features": store_pipeline["feature_scores"],
            },
            neuronpedia_model=NOTEBOOK_CFG.neuronpedia_model,
            neuronpedia_set=NOTEBOOK_CFG.neuronpedia_set,
            neuronpedia_base_url=NOTEBOOK_CFG.neuronpedia_base_url,
        )
        display_topk_token_predictions(
            store_pipeline["rendered_prompt"],
            store_pipeline["pre_logits"],
            store_pipeline["post_logits"],
            tokenizer,
            k=5,
            key_tokens=[
                (label, token_id) for label, token_id in zip(store_pipeline["key_labels"], store_pipeline["key_ids"])
            ],
        )
        display_key_token_logits(
            store_pipeline["pre_logits"],
            store_pipeline["post_logits"],
            store_pipeline["key_ids"],
            store_pipeline["key_labels"],
            title=f"Store {NOTEBOOK_CFG.default_scale_factor}× — Key-Token Logits",
        )

## Phase 4b: Direct Projection (Store Direction)

Bypasses feature selection entirely — adds the scaled store concept direction
vector directly to the residual stream at `unembed.hook_in` (lm_head input).

In [ ]:
if IS_DEBUG_INTERVENTION_MODE:
    print_phase_skip("direct projection is replaced by debug intervention validation.")
else:
    direct_projection_source = store_direction_result if USES_STORE_DIRECTION else embed_direction_result
    direct_projection_label = "DirectProj_Store" if USES_STORE_DIRECTION else "DirectProj_Embed"
    dp_pipeline = _run_direct_projection_pipeline(
        NOTEBOOK_CFG,
        direct_projection_source["direction"],
        direct_projection_label,
        scale_factor=NOTEBOOK_CFG.default_scale_factor,
    )
    RESULTS["dp_pipeline"] = dp_pipeline
    print(f"Direct Projection ({direct_projection_label}, {NOTEBOOK_CFG.default_scale_factor}×):")
    print(f"  Pre gap:  {dp_pipeline['pre_gap']:+.4f}")
    print(f"  Post gap: {dp_pipeline['post_gap']:+.4f}")
    print(f"  Δ gap:    {dp_pipeline['gap_delta']:+.4f}")
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
if IS_DEBUG_INTERVENTION_MODE:
    print_phase_skip("direct projection display is replaced by debug intervention validation.")
else:
    with experiment_session(
        NOTEBOOK_CFG.work_root,
        _phase_run_name(NOTEBOOK_CFG.experiment_name, "dp_display"),
        **NOTEBOOK_CFG.session_kwargs,
    ) as (_, _, tokenizer):
        display_topk_token_predictions(
            dp_pipeline["rendered_prompt"],
            dp_pipeline["pre_logits"],
            dp_pipeline["post_logits"],
            tokenizer,
            k=5,
            key_tokens=[(label, token_id) for label, token_id in zip(dp_pipeline["key_labels"], dp_pipeline["key_ids"])],
        )
        display_key_token_logits(
            dp_pipeline["pre_logits"],
            dp_pipeline["post_logits"],
            dp_pipeline["key_ids"],
            dp_pipeline["key_labels"],
            title=f"Direct Projection {NOTEBOOK_CFG.default_scale_factor}× — Key-Token Logits",
        )

## Phase 6a: Debug Intervention Validation

In [ ]:
if IS_DEBUG_INTERVENTION_MODE:
    debug_validation = _run_debug_intervention_validation(NOTEBOOK_CFG)
    RESULTS["debug_validation"] = debug_validation
    display_debug_intervention_validation_report(debug_validation)

    if not debug_validation["all_passed"]:
        failure_message = (
            "Debug intervention validation exceeded configured tolerances; "
            "see RESULTS['debug_validation']."
        )
        if debug_validation["debug_validation_raise_on_failure"]:
            raise AssertionError(failure_message)
        print(f"WARNING: {failure_message}")
else:
    print_phase_skip("debug intervention validation only applies to debug_intervention_pipelines mode.")

## Phase 6b: Scale Factor Sweep

In [ ]:
if IS_DEBUG_INTERVENTION_MODE:
    print_phase_skip("scale-factor sweep is disabled in debug_intervention_pipelines mode.")
else:
    sweep_results = _run_scale_sweep(
        NOTEBOOK_CFG,
        embed_direction_result["direction"],
        embed_direction_result["group_a_ids"],
        embed_direction_result["group_b_ids"],
    )
    RESULTS["sweep_results"] = sweep_results
    for sweep_result in sweep_results:
        print(f"Scale={sweep_result['scale_factor']:.1f}x Δ={sweep_result['gap_delta']:+.4f}")
    scales = [entry["scale_factor"] for entry in sweep_results]
    deltas = [entry["gap_delta"] for entry in sweep_results]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(scales, deltas, "o-", color="#2471A3", linewidth=2, markersize=8)
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("Scale Factor")
    ax.set_ylabel(f"Gap Δ ({embed_pipeline['target_a_tok']}−{embed_pipeline['target_b_tok']})")
    ax.set_title(
        f"Scale Factor Sweep — {NOTEBOOK_CFG.model_family}:{NOTEBOOK_CFG.model_variant} "
        f"{NOTEBOOK_CFG.analysis_concept_label}"
    )
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    for sweep_result in [sweep_results[0], sweep_results[-1]]:
        display_key_token_logits(
            sweep_result["pre_logits"],
            sweep_result["post_logits"],
            sweep_result["key_ids"],
            sweep_result["key_labels"],
            title=f"Embed {sweep_result['scale_factor']}× — Key-Token Logits",
        )

## Phase 7 and 8: Progressive Ablation and Sign-Aware Feature Selection

In [ ]:
if IS_DEBUG_INTERVENTION_MODE:
    print_phase_skip("progressive ablation and sign-aware analysis are disabled in debug_intervention_pipelines mode.")
else:
    feature_pool = _collect_feature_pool(
        NOTEBOOK_CFG,
        embed_direction_result["direction"],
        embed_direction_result["group_a_ids"],
        embed_direction_result["group_b_ids"],
        top_n=max(max(NOTEBOOK_CFG.ablation_n_list), NOTEBOOK_CFG.top_n * 2),
    )

    feature_count = int(feature_pool["feature_ids"].shape[0])
    eligible_ablation_n = [n_value for n_value in NOTEBOOK_CFG.ablation_n_list if n_value <= feature_count]
    skipped_ablation_n = [n_value for n_value in NOTEBOOK_CFG.ablation_n_list if n_value > feature_count]
    RESULTS["feature_pool_summary"] = {
        "feature_count": feature_count,
        "target_a_id": feature_pool["target_a_id"],
        "target_b_id": feature_pool["target_b_id"],
        "eligible_ablation_n_list": eligible_ablation_n,
        "skipped_ablation_n_list": skipped_ablation_n,
        "constrained_feature_selection_refs": NOTEBOOK_CFG.constrained_feature_selection_refs,
    }
    print(f"Feature pool count: {feature_count}")
    if skipped_ablation_n:
        print(
            "Skipping requested ablation sizes above the available feature count: "
            f"{skipped_ablation_n}"
        )

    ablation_groups, ablation_logit_diffs, ablation_results = _run_ablations(
        NOTEBOOK_CFG,
        feature_pool,
        embed_pipeline["pre_logits"],
    )
    RESULTS["ablation_logit_diffs"] = ablation_logit_diffs

    display_ablation_chart(
        ablation_groups,
        logit_diffs=ablation_logit_diffs,
        title=(
            f"Embed ablation — {NOTEBOOK_CFG.model_family}:{NOTEBOOK_CFG.model_variant} "
            f"{NOTEBOOK_CFG.analysis_concept_label}"
        ),
    )

    if ablation_results:
        strongest_n = max(ablation_results)
        display_key_token_logits(
            embed_pipeline["pre_logits"],
            ablation_results[strongest_n],
            feature_pool["key_ids"],
            feature_pool["key_labels"],
            title=f"Top-{strongest_n} Ablation — Key-Token Logits",
        )

    if NOTEBOOK_CFG.enable_sign_aware:
        sign_aware = _run_sign_aware(NOTEBOOK_CFG, feature_pool, embed_pipeline["pre_logits"])
        RESULTS["sign_aware"] = {
            "positive_feature_count": int(len(sign_aware["positive_features"])),
            "negative_feature_count": int(len(sign_aware["negative_features"])),
            "messages": list(sign_aware.get("messages", [])),
        }
        for message in sign_aware.get("messages", []):
            print(f"\n⚠️  Sign-aware: {message}")
        has_pos = len(sign_aware["positive_features"]) > 0
        has_neg = len(sign_aware["negative_features"]) > 0
        if has_pos or has_neg:
            feature_display = {}
            score_display = {}
            if has_pos:
                feature_display["Positive activation"] = feature_ids_to_tuples(sign_aware["positive_features"][: NOTEBOOK_CFG.top_n])
                score_display["Positive activation"] = tensor_to_cpu(sign_aware["positive_scores"][: NOTEBOOK_CFG.top_n]).tolist()
            if has_neg:
                feature_display["Negative activation"] = feature_ids_to_tuples(sign_aware["negative_features"][: NOTEBOOK_CFG.top_n])
                score_display["Negative activation"] = tensor_to_cpu(sign_aware["negative_scores"][: NOTEBOOK_CFG.top_n]).tolist()
            display_top_features_comparison(
                feature_display,
                score_display,
                neuronpedia_model=NOTEBOOK_CFG.neuronpedia_model,
                neuronpedia_set=NOTEBOOK_CFG.neuronpedia_set,
                neuronpedia_base_url=NOTEBOOK_CFG.neuronpedia_base_url,
            )
        else:
            print("\n⚠️  Sign-aware: No features had non-zero activations — nothing to display.")
        if "positive_post_logits" in sign_aware:
            display_key_token_logits(
                embed_pipeline["pre_logits"],
                sign_aware["positive_post_logits"],
                feature_pool["key_ids"],
                feature_pool["key_labels"],
                title=f"Positive features {NOTEBOOK_CFG.default_scale_factor}× — Key-Token Logits",
            )
        if "negative_post_logits" in sign_aware:
            display_key_token_logits(
                embed_pipeline["pre_logits"],
                sign_aware["negative_post_logits"],
                feature_pool["key_ids"],
                feature_pool["key_labels"],
                title="Negative features ablated — Key-Token Logits",
            )
    else:
        print("\nSign-aware feature analysis: DISABLED (set ENABLE_SIGN_AWARE=true to enable)")

## Phase 9: Direction Consistency Probes

In [ ]:
if not USES_STORE_DIRECTION:
    print_phase_skip("direction consistency probes only apply when both embed and store directions are available.")
else:
    probe_results = _run_direction_probes(NOTEBOOK_CFG, embed_direction_result["direction"], store_direction_result["direction"])
    RESULTS["probe_results"] = probe_results
    display_direction_probe_results(probe_results)

## Phase 9b: Local Graph Upload Artifacts

In [ ]:
graph_artifacts = []
for result_key in ("embed_pipeline", "store_pipeline", "debug_validation"):
    result_payload = RESULTS.get(result_key)
    if isinstance(result_payload, dict):
        artifact = result_payload.get("graph_artifact")
        if artifact:
            graph_artifacts.append({"result_key": result_key, **artifact})

for sweep_index, sweep_result in enumerate(RESULTS.get("sweep_results", [])):
    artifact = sweep_result.get("graph_artifact")
    if artifact:
        graph_artifacts.append({"result_key": f"sweep_results[{sweep_index}]", **artifact})

if graph_artifacts:
    RESULTS["graph_artifacts"] = graph_artifacts
    print(json.dumps(graph_artifacts, indent=2))
else:
    print("Local graph upload artifacts: none recorded for this run.")

## Phase 10: Local Explanation Coverage

In [ ]:
if IS_DEBUG_INTERVENTION_MODE:
    print_phase_skip("local explanation coverage is disabled in debug_intervention_pipelines mode.")
elif NOTEBOOK_CFG.check_local_explanation_coverage or NOTEBOOK_CFG.generate_missing_local_explanations:
    embed_feature_ids = embed_pipeline["feature_ids"][: NOTEBOOK_CFG.local_explanation_feature_limit]
    store_feature_ids = []
    if "store_pipeline" in RESULTS:
        store_feature_ids = store_pipeline["feature_ids"][: NOTEBOOK_CFG.local_explanation_feature_limit]
    explanation_preparation = prepare_local_explanation_backfill(
        NOTEBOOK_CFG,
        embed_feature_ids,
        store_feature_ids,
    )
    prefetch_summary = {
        "export_roots": list(explanation_preparation.export_roots),
        "cache_dir": explanation_preparation.cache_dir,
        "checked_feature_urls": [feature_ref.feature_url for feature_ref in explanation_preparation.feature_refs],
        "missing_feature_urls": [
            status.feature_ref.feature_url
            for status in explanation_preparation.initial_statuses
            if not status.has_local_explanation
        ],
        "cache_ready_feature_urls": [
            status.feature_ref.feature_url
            for status in explanation_preparation.prefetch_statuses
            if status.cache_ready
        ],
        "prefetch_statuses": [
            {
                "feature_url": status.feature_ref.feature_url,
                "explanation_count": status.explanation_count,
                "cache_ready": status.cache_ready,
                "cache_source": status.cache_source,
                "cache_path": status.cache_path,
                "activation_rows": status.activation_rows,
                "error": status.error,
            }
            for status in explanation_preparation.prefetch_statuses
        ],
        "prefetch_failures": [
            {"feature_url": status.feature_ref.feature_url, "error": status.error}
            for status in explanation_preparation.prefetch_statuses
            if not status.cache_ready
        ],
    }
    RESULTS["local_explanation_prefetch"] = prefetch_summary

    explanation_coverage = ensure_local_feature_explanations(
        explanation_preparation.feature_refs,
        generate_missing=NOTEBOOK_CFG.generate_missing_local_explanations,
        local_db_url=NOTEBOOK_CFG.local_neuronpedia_db_url,
        timeout_seconds=NOTEBOOK_CFG.local_explanation_timeout_seconds,
        type_name=NOTEBOOK_CFG.local_explanation_type_name,
        max_retries=NOTEBOOK_CFG.local_explanation_max_retries,
        retry_backoff_seconds=NOTEBOOK_CFG.local_explanation_retry_backoff_seconds,
    )
    coverage_summary = {
        "use_localhost": NOTEBOOK_CFG.use_localhost,
        "neuronpedia_base_url": NOTEBOOK_CFG.neuronpedia_base_url,
        "explanation_type_name": NOTEBOOK_CFG.local_explanation_type_name,
        "timeout_seconds": NOTEBOOK_CFG.local_explanation_timeout_seconds,
        "max_retries": NOTEBOOK_CFG.local_explanation_max_retries,
        "retry_backoff_seconds": NOTEBOOK_CFG.local_explanation_retry_backoff_seconds,
        "checked_feature_urls": [status.feature_ref.feature_url for status in explanation_coverage.statuses],
        "missing_feature_urls": [
            status.feature_ref.feature_url
            for status in explanation_coverage.statuses
            if not status.has_local_explanation
        ],
        "generated_artifact_paths": [str(artifact.artifact_path) for artifact in explanation_coverage.generated_artifacts],
        "generation_failures": [
            {"feature_url": failure.feature_ref.feature_url, "error": failure.error}
            for failure in explanation_coverage.generation_failures
        ],
        "prefetch_summary": prefetch_summary,
    }
    RESULTS["local_explanation_coverage"] = coverage_summary
    display_local_explanation_report(
        prefetch_summary=prefetch_summary,
        coverage_summary=coverage_summary,
    )
else:
    print(
        "Local explanation coverage: DISABLED (set CHECK_LOCAL_EXPLANATION_COVERAGE=true or "
        "GENERATE_MISSING_LOCAL_EXPLANATIONS=true to enable)"
    )

## Cleanup

In [ ]:
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if SHOULD_CLEANUP_WORK_ROOT:
    shutil.rmtree(NOTEBOOK_CFG.work_root, ignore_errors=True)

SUMMARY_RECORD = collect_summary(
    NOTEBOOK_CFG,
    RESULTS,
    config_path=EXPERIMENT_CONFIG_PATH,
    work_root_removed=SHOULD_CLEANUP_WORK_ROOT,
)

print("Notebook complete.")
print(json.dumps(SUMMARY_RECORD, indent=2))